# 04 — Transfer learning and ablations (Tasks 3 and 4)

Fine-tunes ImageNet-pretrained backbones from `timm` on fold 0. The same notebook runs any list of presets from
`src/config.py` through the `RUNS` parameter:

* **Main A/B** (default): `convnext_tiny_plain` vs `convnext_tiny_shift` — ConvNeXt-Tiny at 128 px, identical
  except for the training augmentation (`plain` vs `shift`). About 2 hours per run on the M1 Pro.
* **Ablations** (`RUNS = "abl_base,abl_plain,abl_letterbox,abl_scale04,abl_scale08,abl_nolowres,abl_nosmooth,abl_96,abl_160"`):
  EfficientNet-B0 at 128 px, 6 epochs, each run changes one setting of `abl_base`. About 30–40 minutes per run.
* **Other architectures** for the ensemble: `vit_small_shift`, `effnet_b0_shift`; 224 px presets
  (`convnext_tiny_shift_224`, `vit_small_shift_224`) are meant for a CUDA GPU.

Recipe (all pretrained runs): AdamW (backbone lr 2e-4, head lr 1e-3, weight decay 0.05), 1 warm-up epoch then cosine
decay, label smoothing 0.1, EMA of the weights (decay 0.999) used for validation, early stopping when val_shift accuracy
has not improved for 4 epochs, best checkpoint chosen on val_shift. Each run writes a fold-0 submission (no TTA)
so the leaderboard can be compared with the validation scores.

Long runs are meant to run in the background from a terminal (see README.md), e.g.
`caffeinate -is papermill 04_transfer_learning.ipynb outputs/logs/04_main.ipynb -p RUNS "convnext_tiny_plain,convnext_tiny_shift"`.

In [ ]:
RUNS = "convnext_tiny_plain,convnext_tiny_shift"   # comma-separated preset names
FOLD = 0
FORCE = False              # True = retrain from scratch even if finished
MAKE_SUBMISSIONS = True    # write a fold-FOLD submission (no TTA) for every run

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))  # the notebooks live in the repo root, next to src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

import torch
from sklearn.metrics import confusion_matrix

from src.config import CLASSES, EXPERIMENTS_CSV, PRESETS, SUBMISSIONS_CSV, get_device, make_config, make_dirs
from src.data import load_test_df
from src.engine import fit, get_preds
from src.models import count_params, create_model
from src.plots import plot_confusion, plot_history, savefig
from src.utils import accuracy, softmax, write_submission

make_dirs()
print("device:", get_device())

import timm

runs = [r.strip() for r in RUNS.split(",") if r.strip()]

In [ ]:
def experiments_for(runs, folds=None):
    """Rows of results/experiments.csv for the given runs (and folds)."""
    ex = pd.read_csv(EXPERIMENTS_CSV)
    ex = ex[ex["run"].isin(runs)]
    if folds is not None:
        ex = ex[ex["fold"].isin(folds)]
    return ex.reset_index(drop=True)


def per_class_accuracy(run, fold):
    """Accuracy per true class on val_clean and val_shift for one fold."""
    out = {}
    for split in ["val_clean", "val_shift"]:
        _, logits, labels = get_preds(run, fold, split)
        pred = logits.argmax(1)
        out[split] = [(pred[labels == k] == k).mean() for k in range(len(CLASSES))]
    return pd.DataFrame(out, index=CLASSES)

## 1. All presets and the selected runs

In [ ]:
keys = ["model_name", "img_size", "aug", "resize_mode", "crop_scale_min", "lowres_p", "label_smoothing",
        "epochs", "batch_size", "lr", "head_lr"]
presets = pd.DataFrame({name: {k: getattr(make_config(name), k) for k in keys} for name in PRESETS}).T
presets["selected"] = presets.index.isin(runs)
presets

Pretrained weights used by the selected runs (for `REFERENCES.md`).

In [ ]:
rows = []
for run in runs:
    cfg = make_config(run)
    if cfg.model_name == "simple_cnn":
        continue
    m = create_model(cfg, pretrained=False)
    pc = m.pretrained_cfg
    rows.append({"run": run, "model": cfg.model_name, "params (M)": round(count_params(m) / 1e6, 1),
                 "pretrained on": pc.get("tag", ""), "hf_hub_id": pc.get("hf_hub_id", ""),
                 "original input size": pc.get("input_size"), "mean": pc.get("mean"), "std": pc.get("std"),
                 "paper": pc.get("paper_ids", "")})
pd.DataFrame(rows)

## 2. Training

In [ ]:
histories = {}
for run in runs:
    histories[run] = fit(make_config(run), fold=FOLD, force=FORCE)

## 3. Learning curves

In [ ]:
for run, hist in histories.items():
    plot_history(hist, title=f"{run} (fold {FOLD})", name=f"04_curves_{run}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for run, hist in histories.items():
    axes[0].plot(hist["epoch"], hist["val_clean_acc"], marker="o", label=run)
    axes[1].plot(hist["epoch"], hist["val_shift_acc"], marker="o", label=run)
axes[0].set_title("val_clean accuracy")
axes[1].set_title("val_shift accuracy")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend(fontsize=8)
plt.tight_layout()
savefig("04_val_curves_" + "_".join(runs)[:80])
plt.show()

> **Observations (to be written by me):**

## 4. Comparison

The selected runs, then **every** run finished on this fold so far (all of `results/experiments.csv`).

In [ ]:
cols = ["run", "model", "img_size", "aug", "resize_mode", "crop_scale_min", "lowres_p", "label_smoothing",
        "best_epoch", "epochs_run", "train_acc", "val_clean_acc", "val_shift_acc", "val_shift_loss", "minutes"]
display(experiments_for(runs, [FOLD])[cols])
all_fold = pd.read_csv(EXPERIMENTS_CSV)
all_fold = all_fold[all_fold["fold"] == FOLD].sort_values("val_shift_acc", ascending=False)
display(all_fold[cols])

In [ ]:
plot_df = all_fold.melt(id_vars="run", value_vars=["val_clean_acc", "val_shift_acc"], var_name="split", value_name="accuracy")
fig, ax = plt.subplots(figsize=(max(8, 0.9 * all_fold["run"].nunique()), 4))
sns.barplot(data=plot_df, x="run", y="accuracy", hue="split", ax=ax)
lo = plot_df["accuracy"].min()
ax.set_ylim(max(0, lo - 0.05), 1)
ax.tick_params(axis="x", rotation=45)
ax.set_title(f"All finished runs on fold {FOLD}")
plt.tight_layout()
savefig(f"04_all_runs_fold{FOLD}")
plt.show()

Ablations only: change of each `abl_*` run relative to `abl_base` (positive = better than the base).

In [ ]:
abl = all_fold[all_fold["run"].str.startswith("abl_")].set_index("run")
if "abl_base" in abl.index and len(abl) > 1:
    delta = (abl[["val_clean_acc", "val_shift_acc"]] - abl.loc["abl_base", ["val_clean_acc", "val_shift_acc"]])
    display((delta.drop("abl_base") * 100).round(2).rename(columns=lambda c: c.replace("_acc", " Δ (points)")))
else:
    print("no ablation runs with abl_base finished yet")

In [ ]:
display(pd.concat({run: per_class_accuracy(run, FOLD) for run in runs}, axis=1).round(4))
for run in runs:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for ax, split in zip(axes, ["val_clean", "val_shift"]):
        _, logits, labels = get_preds(run, FOLD, split)
        plot_confusion(confusion_matrix(labels, logits.argmax(1), labels=range(len(CLASSES))), CLASSES,
                       title=f"{run} {split}", ax=ax)
    plt.tight_layout()
    savefig(f"04_confusion_{run}")
    plt.show()

> **Observations (to be written by me):**

## 5. Submissions (fold-0 model, no TTA)

One file per run: `submissions/<run>_fold0.csv`. Upload the ones needed for a planned leaderboard comparison and
type the scores into `results/submissions.csv`.

In [ ]:
if MAKE_SUBMISSIONS:
    test_df = load_test_df()
    ex = experiments_for(runs, [FOLD]).set_index("run")
    for run in runs:
        ids, logits, _ = get_preds(run, FOLD, "test")
        assert (ids == test_df["image_id"].values).all()
        write_submission(test_df, softmax(logits), f"{run}_fold{FOLD}", runs=run,
                         val_clean=ex.loc[run, "val_clean_acc"], val_shift=ex.loc[run, "val_shift_acc"],
                         notes=f"fold {FOLD}, no TTA")
    display(pd.read_csv(SUBMISSIONS_CSV))